# Семинар 2. Считаем ценность на своём лабиринте

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/02-environments/seminar/seminar.ipynb)

Сегодня своими руками напишем всё, что на лекции было показано готовым. Среду брать не нужно — возьмём ту же, что в лекции, и нарисуем в ней свой лабиринт.

| | Что делаем | Раздел лекции |
|---|---|---|
| 1 | Свой лабиринт: рисуем карту и смотрим, как ходит случайный агент | 2 |
| 2 | Модель среды: собираем таблицы $p$ и $r$ | 2.1 |
| 3 | Одна итерация уравнения Беллмана | 4.1 |
| 4 | Value iteration: решаем уравнение до конца | 4.2 |
| 5 | Оценка заданной стратегии | 7 |
| 6 | Один шаг улучшения | 7.1 |
| 7 | Эксперименты: меняем плату за шаг и ветер | 4 |

Пять ячеек с `# TODO` — каждая на одну-пять строк. Под каждой сразу идёт проверка: она должна пройти без ошибок.

In [ ]:
# В Google Colab: скачиваем те же два служебных файла, что использовались на лекции.
# Локально они лежат в соседней папке, поэтому просто добавляем её в путь поиска.
import pathlib, sys, urllib.request
LECTURE = "https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/02-environments/lecture/"
if pathlib.Path("../lecture/gridworld.py").exists():
    sys.path.append("../lecture")
else:
    for name in ["gridworld.py", "gridworld_plots.py"]:
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(LECTURE + name, name)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gridworld import (GridWorld, GAMMA, uniform_policy, greedy_policy, run_episode,
                       average_return, estimate_values_mc, solve_q_star,
                       mdp_matrices as mdp_matrices_reference)      # эталон для проверок; свою версию напишем сами
from gridworld_plots import draw_grid, plot_values, plot_q, draw_policy, draw_paths

## 1. Свой лабиринт

Карта задаётся строками: `.` — свободная клетка, `#` — стена, `S` — старт, `G` — выход (+1), `X` — яма (−1). Правила те же, что на лекции: ветер с вероятностью `noise` сносит вбок, каждый шаг стоит `step_reward`.

Ниже — готовый лабиринт 3 × 5. Поменяйте его на свой: добавьте стен, передвиньте яму, сделайте поле больше. **Важно:** проверьте, что выход достижим — если он со всех сторон окружён стенами и ямой, агенту останется только тонуть, и все дальнейшие числа будут отрицательными. Проверка ниже это поймает.

In [ ]:
layout = [
    "....G",
    ".##.X",
    "S....",
]
env = GridWorld(layout=layout, step_reward=-0.04, noise=0.1)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
draw_grid(env, ax=axes[0], numbers=True); axes[0].set_title("карта: номера клеток", fontsize=10)
rng = np.random.default_rng(0)
draw_paths([run_episode(env, uniform_policy(env), rng) for _ in range(30)], env,
           ax=axes[1], title="30 эпизодов случайного агента")
plt.show()

# Проверка 1: выход достижим, случайный агент иногда доходит.
goal = next(s for s in range(env.n_states) if env.char(s) == "G")
rng = np.random.default_rng(0)
reached = np.mean([run_episode(env, uniform_policy(env), rng)[0][-1] == goal for _ in range(300)])
assert reached > 0.02, "случайный агент ни разу не дошёл до выхода — проверьте, что до него есть дорога"
print(f"OK: клеток {env.n_states}, старт {env.start}, выход {goal}; случайный агент доходит в {reached:.0%} эпизодов")
print(f"средний return случайной стратегии: {average_return(env, uniform_policy(env), np.random.default_rng(1)):+.2f}")

## 2. Модель среды: таблицы $p$ и $r$

Как на лекции (раздел 2.1), вся среда — это два массива:

* `P[s, a, s_next]` — вероятность попасть в клетку `s_next`, выбрав действие `a` в клетке `s`;
* `R[s, a]` — средняя награда за такой шаг.

Собрать их можно, не запуская среду: метод `env.transitions(s, a)` возвращает список исходов, по одному кортежу `(вероятность, следующая клетка, награда, конец эпизода)` на исход. Например:

```python
>>> env.transitions(10, 3)
[(0.8, 5, -0.04, False), (0.1, 10, -0.04, False), (0.1, 11, -0.04, False)]
```

Пройдите по всем клеткам и действиям и разложите эти исходы по двум массивам. Помните, что в `R[s, a]` попадает **средняя** награда: каждую награду нужно домножить на вероятность её исхода.

In [ ]:
def mdp_matrices(env):
    """Модель среды: P[s, a, s_next] — вероятности переходов, R[s, a] — средняя награда за шаг."""
    P = np.zeros((env.n_states, env.n_actions, env.n_states))
    R = np.zeros((env.n_states, env.n_actions))
    for s in range(env.n_states):
        for a in range(env.n_actions):
            # TODO: пройти по env.transitions(s, a) и заполнить P[s, a, s_next] и R[s, a]
            raise NotImplementedError
    return P, R

# Проверка 2
P, R = mdp_matrices(env)
assert np.allclose(P.sum(axis=2), 1.0), "из каждой пары (клетка, действие) вероятности должны суммироваться в 1"
P_ref, R_ref = mdp_matrices_reference(env)
assert np.allclose(P, P_ref) and np.allclose(R, R_ref), "таблицы не совпали с эталоном"
print(f"OK: P имеет форму {P.shape}, R — {R.shape}")
print(f"P[{env.start}, ↑] ненулевые:", {int(s): round(float(P[env.start, 3, s]), 2) for s in np.flatnonzero(P[env.start, 3])})

## 3. Одна итерация уравнения Беллмана

Формула из раздела 4.1 лекции:

$$
V_k(s') = \max_{a'} Q_k(s', a'), \qquad
Q_{k+1}(s, a) = r(s, a) + \gamma \sum_{s'} p(s' \mid s, a)\, V_k(s').
$$

Напишите её в векторном виде — это две строки. Проверка сравнит ваш результат с той же формулой, расписанной явными циклами.

In [ ]:
def bellman_update(Q, P, R, gamma=GAMMA):
    """Одна итерация: подставляем приближение Q в правую часть уравнения оптимальности."""
    # TODO: 1) V = лучшее число в строке каждой клетки;  2) Q_new = R + gamma * (P @ V)
    raise NotImplementedError

def bellman_update_loops(Q, P, R, gamma=GAMMA):
    """Та же формула, расписанная циклами — дана для проверки."""
    V, Q_new = Q.max(axis=1), np.zeros_like(Q)
    for s in range(env.n_states):
        for a in range(env.n_actions):
            Q_new[s, a] = R[s, a] + gamma * sum(P[s, a, s_next] * V[s_next] for s_next in range(env.n_states))
    return Q_new

# Проверка 3
Q_zero = np.zeros((env.n_states, env.n_actions))
for Q_test in (Q_zero, np.random.default_rng(0).normal(size=Q_zero.shape)):
    assert np.allclose(bellman_update(Q_test, P, R), bellman_update_loops(Q_test, P, R)), "ваша итерация не совпала с формулой"
print("OK: одна итерация из нулей, клетка рядом с выходом:", bellman_update(Q_zero, P, R).max(axis=1).round(2)[:5])

## 4. Value iteration

Теперь повторяем итерацию, пока приближение не перестанет меняться (раздел 4.2 лекции). Условие остановки: наибольшая поправка меньше `tol`.

In [ ]:
def value_iteration(P, R, gamma=GAMMA, tol=1e-10, max_iter=1000):
    """Решение уравнения оптимальности. Возвращает Q* и число сделанных итераций."""
    Q = np.zeros((env.n_states, env.n_actions))
    for k in range(1, max_iter + 1):
        # TODO: посчитать новое приближение через bellman_update; если оно почти не отличается
        #       от Q (наибольшая разница меньше tol) — вернуть его и k; иначе продолжить
        raise NotImplementedError
    return Q, max_iter

# Проверка 4
Q_star, n_iter = value_iteration(P, R)
V_star = Q_star.max(axis=1)
pi_star = greedy_policy(Q_star)
assert np.allclose(Q_star, solve_q_star(P, R), atol=1e-6), "решение не совпало с эталоном"
assert average_return(env, pi_star, np.random.default_rng(0), 300) > average_return(env, uniform_policy(env), np.random.default_rng(0), 300)
print(f"OK: сошлось за {n_iter} итераций; ценность старта {V_star[env.start]:+.2f}")
print(f"средний return: жадная стратегия {average_return(env, pi_star, np.random.default_rng(0), 300):+.2f}, "
      f"случайная {average_return(env, uniform_policy(env), np.random.default_rng(0), 300):+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
plot_q(Q_star, env, ax=axes[0], title="Q*(s, a)")
draw_policy(pi_star, env, ax=axes[1], values=V_star, title="V* и оптимальная стратегия")
plt.show()

**Обсудите.** Посмотрите на стрелки: где агент идёт в обход ямы, а где рискует? Найдите клетку, в которой ответ был бы другим, если бы ветра не было (`noise=0`).

## 5. Оценка заданной стратегии

До сих пор мы искали **лучшую** стратегию. Теперь оценим **любую** — для этого в уравнении вместо максимума стоит усреднение по стратегии (раздел 7 лекции):

$$
V^\pi(s') = \sum_{a'} \pi(a' \mid s')\, Q^\pi(s', a'), \qquad
Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} p(s' \mid s, a)\, V^\pi(s').
$$

Отличие от раздела 3 — одна строка: вместо `Q.max(axis=1)` берём `(policy * Q).sum(axis=1)`.

In [ ]:
def evaluate_policy(policy, P, R, gamma=GAMMA, n_iter=300):
    """Ценность заданной стратегии. Возвращает Q^π и V^π."""
    Q = np.zeros((env.n_states, env.n_actions))
    for _ in range(n_iter):
        # TODO: V = среднее по стратегии из Q;  Q = R + gamma * (P @ V)
        raise NotImplementedError
    return Q, (policy * Q).sum(axis=1)

# Проверка 5: ценность случайной стратегии, посчитанная уравнением, совпадает с Монте-Карло
uniform = uniform_policy(env)
Q_uniform, V_uniform = evaluate_policy(uniform, P, R)

rng = np.random.default_rng(1)
V_mc = estimate_values_mc([run_episode(env, uniform, rng) for _ in range(3000)], env.n_states)
visited = ~np.isnan(V_mc)
assert abs(V_uniform[env.start] - V_mc[env.start]) < 0.05, "ценность старта разошлась с Монте-Карло"
assert np.abs(V_uniform[visited] - V_mc[visited]).max() < 0.15, "оценка сильно разошлась с Монте-Карло"
print(f"OK: наибольшее расхождение с Монте-Карло {np.abs(V_uniform[visited] - V_mc[visited]).max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
plot_values(V_mc, env, ax=axes[0], title="3000 эпизодов Монте-Карло")
plot_values(V_uniform, env, ax=axes[1], title="решение уравнения")
plt.show()

## 6. Один шаг улучшения

Имея $Q^\pi$, построим новую стратегию: в каждой клетке выбираем действие с наибольшим $Q^\pi$. Проверка убедится, что ценность не упала **ни в одной** клетке — это и есть гарантия шага улучшения из лекции.

In [ ]:
def improve(policy, P, R, gamma=GAMMA):
    """Один шаг улучшения: оценить стратегию и пересесть на жадные действия по Q^π."""
    # TODO: посчитать Q^π через evaluate_policy и вернуть greedy_policy(Q^π)
    raise NotImplementedError

# Проверка 6
improved = improve(uniform, P, R)
Q_improved, V_improved = evaluate_policy(improved, P, R)
assert np.all(V_improved >= V_uniform - 1e-9), "после шага улучшения ценность не должна падать ни в одной клетке"
assert average_return(env, improved, np.random.default_rng(2), 300) > average_return(env, uniform, np.random.default_rng(2), 300)
print(f"OK: ценность старта {V_uniform[env.start]:+.2f} → {V_improved[env.start]:+.2f} "
      f"(оптимум {V_star[env.start]:+.2f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
draw_policy(uniform, env, ax=axes[0], values=V_uniform, title="было: случайная стратегия")
draw_policy(improved, env, ax=axes[1], values=V_improved, title="стало: жадная по Q^π")
plt.show()

**Обсудите.** Сколько шагов улучшения нужно на вашем лабиринте, чтобы дойти до оптимума? Проверьте: примените `improve` ещё раз к полученной стратегии и сравните с `pi_star`.

## 7. Эксперименты

Код писать не нужно — меняйте числа и смотрите, что происходит. Ячейка ниже строит оптимальную стратегию для нескольких значений платы за шаг и силы ветра.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11.5, 6))
for ax, step_cost in zip(axes[0], [-0.01, -0.1, -0.5]):
    e = GridWorld(layout=layout, step_reward=step_cost, noise=0.1)
    Pe, Re = mdp_matrices(e)
    draw_policy(greedy_policy(value_iteration(Pe, Re)[0]), e, ax=ax, title=f"плата за шаг {step_cost}")
for ax, noise in zip(axes[1], [0.0, 0.1, 0.3]):
    e = GridWorld(layout=layout, step_reward=-0.04, noise=noise)
    Pe, Re = mdp_matrices(e)
    draw_policy(greedy_policy(value_iteration(Pe, Re)[0]), e, ax=ax, title=f"ветер {noise}")
plt.tight_layout(); plt.show()

**Обсудите.**

1. Посмотрите на нижний ряд в верхних трёх картах: при дешёвом шаге агент идёт в обход, при дорогом — напрямик. Почему дорогое время делает риск выгодным?
2. Что происходит при `noise = 0`? Почему стратегия становится «короткий путь любой ценой»?
3. При сильном ветре агент местами ходит «в стену». Зачем?

## 8. Что дальше

* **ДЗ** (`../homework/homework.ipynb`): среда «управление запасами» как `gym.Env`, её модель из распределения спроса, оптимальная стратегия заказов через уравнение оптимальности и теория.
* **Неделя 3**: всё то же самое, но когда модель среды **неизвестна** — Monte-Carlo, TD-обучение, SARSA и Q-learning; и policy iteration целиком, а не один шаг улучшения.